# Speaker Identification — Final Pipeline (Clean)


1. Setup & configuration  
2. Data loading & label normalization  
3. LLM inference (zero-shot / few-shot)  
4. Post-processing  
5. Evaluation (Accuracy, Macro-F1, Weighted-F1, per-class report, confusion matrix)  
6. Error analysis + examples  


## 1) Setup
Install deps (optional in Colab), and load environment variables safely. **Do not hardcode API keys.**

In [ ]:
# Optional (Colab / fresh env)
# !pip install -U openai pandas numpy tqdm scikit-learn python-dotenv

import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# If you use a .env file:
# from dotenv import load_dotenv
# load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Missing OPENAI_API_KEY env var"


## 2) Data loading
Expected columns (recommended):
- `case_id` (optional)
- `paragraph_text`
- `sentence_text`
- `speaker1` (gold label)

You can keep additional columns; they will be preserved.

In [ ]:
DATA_PATH = "speaker_anotation_csv.csv"  # <-- update if needed

df = pd.read_csv(DATA_PATH)

# Basic sanity checks
required = {"sentence_text", "speaker1"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

df.head()


## 3) Label normalization
Unify label spelling (e.g., 'נאשם זה הגנה!!' -> 'Defense') so evaluation is meaningful.

In [ ]:
LABEL_MAP = {
    # Hebrew / noisy variants -> canonical English labels
    "שופט": "Judge",
    "תביעה": "Prosecution",
    "מאשימה": "Prosecution",
    "הגנה": "Defense",
    "נאשם": "Defense",          # if in your scheme 'defendant' sentences are defense-side
    "נאשם זה הגנה!!": "Defense",
    "שירות המבחן": "Probation",
    # add any variants you saw in the data:
}

def normalize_label(x: str) -> str:
    if pd.isna(x):
        return x
    x = str(x).strip()
    return LABEL_MAP.get(x, x)

df["speaker_gold"] = df["speaker1"].apply(normalize_label)

# Show label distribution
df["speaker_gold"].value_counts(dropna=False)


## 4) Prompt
We classify **one sentence** while seeing its **full paragraph** as context.

Tips:
- Ask the model to *read the whole paragraph first*, then classify the target sentence.
- Force a **single-label output** from a closed set.
- Add an `Unknown` option for ambiguous cases.

In [ ]:
import json, re, time
from openai import OpenAI

client = OpenAI()

ALLOWED = {"שופט", "הגנה", "תביעה", "שירות המבחן"}

SYSTEM_PROMPT = (
    "אתה מסווג מי הדובר/הטוען של הטענה במשפט הנתון.\n"
    "לפני הסיווג, קרא והבן את הפסקה המלאה כדי לזהות מי הדובר בה.\n"
    "רק לאחר מכן סווג את המשפט שבתוכה.\n"
    "אם יש סתירה בין ניסוח המשפט הבודד לבין ההקשר של הפסקה – ההקשר של הפסקה גובר.\n"
    "בחר בדיוק אחד מתוך: שופט, הגנה, תביעה, שירות המבחן.\n"
    "סווג לפי מי שמביע את הטענה, גם אם מדובר בציטוט, תיאור או סיכום בתוך דברי השופט.\n"
    "החזר רק JSON תקין ללא טקסט נוסף."
)


# ======================
# דוגמאות קשות (Stage B)
# ======================
HARD_EXAMPLES = """
Example 1:
PARAGRAPH: במסגרת טיעוניה לעונש עמדה המאשימה על כך שהערך החברתי בו פגע הנאשם הוא זכותו של אדם לשלמות גופו.
SENTENCE: הערך החברתי בו פגע הנאשם הוא זכותו של אדם לשלמות גופו.
LABEL: תביעה

Example 2:
PARAGRAPH: לאחר בחינת כלל נסיבות המקרה, סבורני כי מעשיו של הנאשם יצרו פוטנציאל ממשי לפגיעה בשלום הציבור.
SENTENCE: מעשיו של הנאשם יצרו פוטנציאל ממשי לפגיעה בשלום הציבור.
LABEL: שופט

Example 3:
PARAGRAPH: צויין בחוות דעת הממונה כי הנאשם הודה בשימוש בסמים, אולם קביעה זו נבחנה על ידי בית המשפט.
SENTENCE: צויין בחוות דעת הממונה כי הנאשם הודה בשימוש בסמים.
LABEL: שופט

Example 4:
PARAGRAPH: מן הראיות עלה כי גורמים שונים נקטו איומים כלפי הנאשם עובר לאירוע.
SENTENCE: גורמים שונים נקטו איומים כלפי הנאשם עובר לאירוע.
LABEL: שופט
"""

def extract_json_obj(text: str) -> dict:
    try:
        return json.loads(text)
    except:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise ValueError(f"No JSON found. Raw: {text[:200]}")
        return json.loads(m.group(0))

def gpt_one(paragraph, sentence, model="gpt-4.1", temperature=0.2, max_retries=4):
    user_prompt = (
        "להלן דוגמאות מתוייגות להבהרת כללי הסיווג:\n"
        f"{HARD_EXAMPLES}\n\n"
        "כעת סווג את הדוגמה הבאה.\n"
        "החזר *רק* אובייקט JSON תקין בפורמט המדויק הבא, בלי שום טקסט נוסף:\n"
        "{\"label\":\"שופט\"} או {\"label\":\"הגנה\"} או {\"label\":\"תביעה\"} או {\"label\":\"שירות המבחן\"}\n\n"
        f"PARAGRAPH: \"\"\"{paragraph}\"\"\"\n"
        f"SENTENCE: \"\"\"{sentence}\"\"\"\n"
    )

    last_err = None
    for i in range(max_retries):
        try:
            resp = client.responses.create(
                model=model,
                input=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=temperature,
            )
            data = extract_json_obj(resp.output_text)
            label = str(data.get("label", "")).strip()
            if label not in ALLOWED:
                raise ValueError(f"Bad label: {label} | raw: {resp.output_text[:200]}")
            return label
        except Exception as e:
            last_err = e
            time.sleep(0.7 * (i + 1))
    raise last_err



## 5) LLM inference
This cell is written so you can swap providers/models easily. Fill in your chosen model name.

If you already have predictions saved (CSV), you can skip to Evaluation.

In [ ]:
MODEL_NAME = "gpt-4.1"  # same default as original notebook
TEMPERATURE = 0.2

# Expected columns: sentence_text, paragraph_text (optional)
# Returns: a copy with pred_label

def run_inference(df, paragraph_col="paragraph_text", sentence_col="sentence_text"):
    df = df.copy()
    preds = []
    for _, row in df.iterrows():
        paragraph = row.get(paragraph_col, "") if paragraph_col in df.columns else ""
        sentence = row[sentence_col]
        label = gpt_one(paragraph=paragraph, sentence=sentence, model=MODEL_NAME, temperature=TEMPERATURE)
        preds.append(label)
    df["pred_label"] = preds
    return df

## 6) Evaluation
Loads `preds_zero_shot.csv` if predictions are already saved, and computes:
- Accuracy
- Macro-F1
- Weighted-F1
- Per-class report
- Confusion matrix

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import re

PREDS_PATH = "preds_zero_shot.csv"  # <-- update if needed

preds_df = pd.read_csv(PREDS_PATH)

# Normalize both gold and predicted labels
preds_df["speaker_gold"] = preds_df["speaker1"].apply(normalize_label)
preds_df["pred_norm"] = preds_df["pred_label"].apply(normalize_label)

y_true = preds_df["speaker_gold"].astype(str)
y_pred = preds_df["pred_norm"].astype(str)

acc = accuracy_score(y_true, y_pred)
macro = f1_score(y_true, y_pred, average="macro")
weighted = f1_score(y_true, y_pred, average="weighted")

print(f"Accuracy:   {acc:.4f}")
print(f"Macro-F1:   {macro:.4f}")
print(f"Weighted-F1:{weighted:.4f}\n")

labels_sorted = sorted(set(y_true) | set(y_pred))
print("Labels:", labels_sorted, "\n")
print(classification_report(y_true, y_pred, labels=labels_sorted, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)
cm


In [ ]:
# Confusion matrix plot (no custom colors)
import numpy as np

labels_sorted = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)

plt.figure(figsize=(8,6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix")
plt.xticks(range(len(labels_sorted)), labels_sorted, rotation=45, ha="right")
plt.yticks(range(len(labels_sorted)), labels_sorted)
plt.xlabel("Predicted")
plt.ylabel("Gold")
plt.tight_layout()
plt.show()


## 7) Error analysis
Show top mismatches and allow manual inspection.

In [ ]:
errors = preds_df[preds_df["pred_norm"] != preds_df["speaker_gold"]].copy()
print("Num errors:", len(errors))

cols = [c for c in ["case_id", "speaker_gold", "pred_norm", "sentence_text", "paragraph_text"] if c in errors.columns]
errors[cols].head(20)
